# Nifty 50 Signal — Training v3 (Tier 1 features + walk-forward)

**v2 outcome:** 45% out-of-sample, confidence anti-correlated with accuracy.

**v3 changes:**
1. **25 features** (was 14): lagged returns 1d/3d/5d/10d/20d, vol regime, real volume features (Upstox), gap %, distance from N-day high/low, ADX trend strength, RSI divergence
2. **Multi-timeframe context**: hourly aggregates merged into each daily row
3. **Walk-forward CV** (5 rolling folds) — honest cross-regime accuracy
4. **Proper stacking** — out-of-fold base predictions feed the meta (no leakage)
5. **6 baselines** compared — model must beat random + always-long + momentum
6. **Held-out test** — last 3 months untouched until final evaluation
7. **Probability calibration** (Platt scaling) — fixes confidence ≠ accuracy

Setup: Runtime → T4 GPU. CSVs in `/MyDrive/nifty_signal/data/` from the Upstox fetch script.

In [ ]:
!pip install -q ta xgboost lightgbm catboost scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/nifty_signal/data'
MODELS_DIR = '/content/drive/MyDrive/nifty_signal/models'
os.makedirs(MODELS_DIR, exist_ok=True)
print('Files:', os.listdir(DATA_DIR))

## Tier 1 feature engineering — 25 features

In [ ]:
import pandas as pd, numpy as np
from ta.momentum import RSIIndicator
from ta.trend import MACD, EMAIndicator, ADXIndicator
from ta.volatility import AverageTrueRange, BollingerBands

def load_csv(path):
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    for c in ('date','timestamp','time'):
        if c in df.columns and 'datetime' not in df.columns:
            df = df.rename(columns={c:'datetime'})
            break
    df['datetime'] = pd.to_datetime(df['datetime'])
    df = df.sort_values('datetime').reset_index(drop=True)
    return df

def add_tier1_features(df):
    out = df.copy()
    c, h, l, v = out['close'], out['high'], out['low'], out['volume']
    
    # --- Classical indicators ---
    out['rsi_14'] = RSIIndicator(c, 14).rsi()
    out['macd_hist'] = MACD(c, 26, 12, 9).macd_diff()
    out['ema_9'] = EMAIndicator(c, 9).ema_indicator()
    out['ema_21'] = EMAIndicator(c, 21).ema_indicator()
    out['ema_50'] = EMAIndicator(c, 50).ema_indicator()
    out['ema_cross_9_21'] = (out['ema_9'] > out['ema_21']).astype(int)
    out['ema_cross_21_50'] = (out['ema_21'] > out['ema_50']).astype(int)
    out['atr_14'] = AverageTrueRange(h, l, c, 14).average_true_range()
    bb = BollingerBands(c, 20, 2)
    out['bb_pct_b'] = (c - bb.bollinger_lband()) / (bb.bollinger_hband() - bb.bollinger_lband())
    
    # --- Tier 1: Lagged returns (momentum) ---
    out['ret_1d'] = c.pct_change(1)
    out['ret_3d'] = c.pct_change(3)
    out['ret_5d'] = c.pct_change(5)
    out['ret_10d'] = c.pct_change(10)
    out['ret_20d'] = c.pct_change(20)
    
    # --- Tier 1: Volatility regime ---
    atr_50 = AverageTrueRange(h, l, c, 50).average_true_range()
    out['vol_regime'] = out['atr_14'] / atr_50
    out['atr_pct'] = out['atr_14'] / c
    
    # --- Tier 1: Volume features (graceful if volume == 0 for an index) ---
    v_safe = v.replace(0, np.nan)
    out['log_vol'] = np.log1p(v_safe).fillna(0)
    vol_ma20 = v_safe.rolling(20).mean()
    vol_std20 = v_safe.rolling(20).std()
    out['vol_vs_ma20'] = (v_safe / vol_ma20).fillna(1.0)        # neutral = 1 when no volume
    out['vol_zscore'] = ((v_safe - vol_ma20) / vol_std20).fillna(0.0)  # neutral = 0
    
    # --- Tier 1: Gap ---
    prev_close = c.shift(1)
    out['gap_pct'] = (out['open'] - prev_close) / prev_close
    
    # --- Tier 1: Distance from N-day high/low ---
    high_20 = h.rolling(20).max()
    low_20 = l.rolling(20).min()
    out['dist_high_20'] = (c - high_20) / high_20
    out['dist_low_20'] = (c - low_20) / low_20
    out['pos_in_range_20'] = (c - low_20) / (high_20 - low_20).replace(0, np.nan)
    
    # --- Tier 1: Trend strength ---
    out['adx_14'] = ADXIndicator(h, l, c, 14).adx()
    
    # --- Tier 1: RSI divergence ---
    price_chg_14 = c.pct_change(14)
    rsi_chg_14 = out['rsi_14'].diff(14)
    out['rsi_divergence'] = np.sign(price_chg_14) * np.sign(rsi_chg_14)
    out['rsi_divergence'] = out['rsi_divergence'].fillna(0)
    
    # --- Calendar ---
    dt = pd.to_datetime(out['datetime'])
    out['day_of_week'] = dt.dt.dayofweek
    out['dte'] = (3 - dt.dt.dayofweek) % 7
    
    return out

FEATURES_DAILY = [
    'rsi_14','macd_hist','ema_cross_9_21','ema_cross_21_50','atr_14','bb_pct_b',
    'ret_1d','ret_3d','ret_5d','ret_10d','ret_20d',
    'vol_regime','atr_pct',
    'log_vol','vol_vs_ma20','vol_zscore',
    'gap_pct',
    'dist_high_20','dist_low_20','pos_in_range_20',
    'adx_14','rsi_divergence',
    'day_of_week','dte',
]
print(f'{len(FEATURES_DAILY)} daily features defined')

## Multi-timeframe context: aggregate hourly into per-day features

In [ ]:
def hourly_to_daily_features(hourly_df):
    """For each trading day, summarise the day's hourly bars into ~5 features.
    These describe intraday character — was it a trend day, choppy, volume-heavy?"""
    h = hourly_df.copy()
    h['date'] = pd.to_datetime(h['datetime']).dt.date
    h['intraday_ret'] = (h['close'] - h['open']) / h['open']
    h['hl_range'] = (h['high'] - h['low']) / h['open']
    
    grouped = h.groupby('date').agg(
        intraday_total_return=('close', lambda s: (s.iloc[-1] - s.iloc[0]) / s.iloc[0] if len(s) > 1 else 0),
        intraday_max_drawdown=('close', lambda s: (s.min() - s.iloc[0]) / s.iloc[0] if len(s) > 1 else 0),
        intraday_vol=('intraday_ret', 'std'),
        intraday_volume_total=('volume', 'sum'),
        intraday_n_bars=('close', 'count'),
    ).reset_index()
    
    grouped['intraday_volume_total'] = np.log1p(grouped['intraday_volume_total'].fillna(0))
    grouped['intraday_vol'] = grouped['intraday_vol'].fillna(0)
    grouped['date'] = pd.to_datetime(grouped['date'])
    return grouped

FEATURES_INTRADAY = ['intraday_total_return','intraday_max_drawdown','intraday_vol','intraday_volume_total','intraday_n_bars']
print(f'{len(FEATURES_INTRADAY)} intraday-aggregate features defined')

## Build the training dataset

In [ ]:
# Load and feature-engineer daily — with explicit tz-stripping and step-by-step diagnostics
daily = load_csv(f'{DATA_DIR}/nifty50_daily.csv')
print(f'[1] daily loaded: {len(daily)} rows')

feats = add_tier1_features(daily)
print(f'[2] features added: {len(feats)} rows, {len(feats.columns)} cols')

# Strip timezone so all date comparisons are clean
feats['date'] = pd.to_datetime(feats['datetime']).dt.tz_localize(None).dt.normalize()
print(f'[3] date column: dtype={feats["date"].dtype}, range {feats["date"].min()} to {feats["date"].max()}')

# Merge intraday aggregates if hourly CSV exists
try:
    hourly = load_csv(f'{DATA_DIR}/nifty50_hourly.csv')
    print(f'[4] hourly loaded: {len(hourly)} rows')
    intraday = hourly_to_daily_features(hourly)
    intraday['date'] = pd.to_datetime(intraday['date']).dt.tz_localize(None).dt.normalize()
    print(f'[5] intraday aggregates: {len(intraday)} day-rows, dtype={intraday["date"].dtype}')
    feats = feats.merge(intraday, on='date', how='left')
    matched = feats[FEATURES_INTRADAY[0]].notna().sum()
    print(f'[6] merged: {len(feats)} rows total, {matched} matched with intraday')
    for col in FEATURES_INTRADAY:
        feats[col] = feats[col].fillna(0)
    ALL_FEATURES = FEATURES_DAILY + FEATURES_INTRADAY
    print(f'[7] using {len(ALL_FEATURES)} features (daily + intraday)')
except Exception as e:
    print(f'[!] Hourly merge skipped: {type(e).__name__}: {e}')
    ALL_FEATURES = FEATURES_DAILY
    print(f'[7] using {len(ALL_FEATURES)} features (daily only)')

# Sanitize: replace +/-inf with NaN so dropna catches them (XGBoost rejects inf)
inf_per_feat = np.isinf(feats[ALL_FEATURES].astype(float)).sum()
if inf_per_feat.sum() > 0:
    print(f'[7b] inf counts (will be replaced with NaN):')
    print(inf_per_feat[inf_per_feat > 0].to_string())
feats[ALL_FEATURES] = feats[ALL_FEATURES].replace([np.inf, -np.inf], np.nan)

# Add forward-looking target
feats['next_return'] = feats['close'].pct_change().shift(-1)
n_before = len(feats)
feats = feats.dropna(subset=['next_return']).reset_index(drop=True)
print(f'[8] after dropna(next_return): {len(feats)} rows (lost {n_before - len(feats)})')

# Per-feature NaN audit
nan_counts = feats[ALL_FEATURES].isna().sum().sort_values(ascending=False)
print(f'[9] NaN counts per feature (top 10):')
print(nan_counts.head(10).to_string())

n_before = len(feats)
feats = feats.dropna(subset=ALL_FEATURES).reset_index(drop=True)
print(f'[10] after dropna(features): {len(feats)} rows (lost {n_before - len(feats)})')

# Final safety: clip absurd outliers (e.g., 0.001% of features that survived dropna
# but have extreme values from division by tiny denominators)
for col in ALL_FEATURES:
    if feats[col].dtype.kind in 'fc':  # float / complex
        q01, q99 = feats[col].quantile([0.001, 0.999])
        feats[col] = feats[col].clip(q01, q99)

if len(feats) == 0:
    raise RuntimeError('Dataset is empty after dropna. Inspect step [9] above to see which features have all-NaN values.')

print(f'\n=== Final dataset: {len(feats)} rows, {feats["date"].min().date()} to {feats["date"].max().date()} ===')
print(f'    Feature range check — any remaining inf? {np.isinf(feats[ALL_FEATURES].astype(float)).any().any()}')

In [ ]:
# Quick sanity check on feature ranges
feats[ALL_FEATURES].describe().T[['mean','std','min','max']]

## Hold out the last 3 months — the honesty test

In [ ]:
HOLDOUT_DAYS = 90
cutoff = feats['date'].max() - pd.Timedelta(days=HOLDOUT_DAYS)
df_train = feats[feats['date'] <= cutoff].reset_index(drop=True)
df_holdout = feats[feats['date'] > cutoff].reset_index(drop=True)
print(f'Cutoff date: {cutoff.date()}')
print(f'Training pool: {len(df_train)} rows ({df_train["date"].min().date()} to {df_train["date"].max().date()})')
print(f'Holdout test:  {len(df_holdout)} rows ({df_holdout["date"].min().date()} to {df_holdout["date"].max().date()})')

assert len(df_train) > 100, f'Training set too small: {len(df_train)} rows. Check date logic above.'
assert len(df_holdout) > 20, f'Holdout too small: {len(df_holdout)} rows. Reduce HOLDOUT_DAYS or check data.'

def make_labels(next_returns, q_low=0.33, q_high=0.67):
    """Compute quantile labels using thresholds from THIS fold's training portion only."""
    lo, hi = next_returns.quantile(q_low), next_returns.quantile(q_high)
    lab = pd.Series(1, index=next_returns.index)  # NO_TRADE = 1
    lab[next_returns > hi] = 2   # BUY_CALL
    lab[next_returns < lo] = 0   # BUY_PUT
    return lab, lo, hi

## Baselines — your model must beat all of these

In [ ]:
from sklearn.metrics import accuracy_score, precision_score

def baseline_random(n, seed=42):
    rng = np.random.default_rng(seed)
    return rng.integers(0, 3, n)

def baseline_always(n, cls):
    return np.full(n, cls, dtype=int)

def baseline_momentum(prev_returns):
    """Predict same direction as previous day's return."""
    return np.where(prev_returns > 0, 2, np.where(prev_returns < 0, 0, 1))

def baseline_mean_reversion(prev_returns):
    """Predict opposite of previous day's return."""
    return np.where(prev_returns > 0, 0, np.where(prev_returns < 0, 2, 1))

def score_baselines(y_true, prev_returns_for_baseline):
    n = len(y_true)
    results = {}
    for name, preds in [
        ('Random',            baseline_random(n)),
        ('Always BUY_CALL',   baseline_always(n, 2)),
        ('Always BUY_PUT',    baseline_always(n, 0)),
        ('Always NO_TRADE',   baseline_always(n, 1)),
        ('Momentum',          baseline_momentum(prev_returns_for_baseline)),
        ('Mean-Reversion',    baseline_mean_reversion(prev_returns_for_baseline)),
    ]:
        acc = accuracy_score(y_true, preds)
        # Precision on directional (non-NO_TRADE) predictions
        dir_mask = preds != 1
        if dir_mask.sum() > 0:
            dir_prec = (preds[dir_mask] == y_true[dir_mask]).mean()
        else:
            dir_prec = float('nan')
        results[name] = {'accuracy': acc, 'directional_precision': dir_prec, 'n_directional': int(dir_mask.sum())}
    return results

## Walk-forward CV — 5 rolling folds

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import joblib

def make_models():
    return {
        'xgb': XGBClassifier(n_estimators=500, max_depth=5, learning_rate=0.04,
                             objective='multi:softprob', num_class=3,
                             eval_metric='mlogloss', tree_method='hist', device='cuda',
                             reg_alpha=0.1, reg_lambda=1.0, subsample=0.8, colsample_bytree=0.8),
        'lgbm': LGBMClassifier(n_estimators=500, learning_rate=0.04, num_leaves=31,
                               objective='multiclass', num_class=3,
                               class_weight='balanced', verbose=-1,
                               reg_alpha=0.1, reg_lambda=1.0, subsample=0.8, colsample_bytree=0.8),
        'cat': CatBoostClassifier(iterations=500, depth=6, learning_rate=0.04,
                                  loss_function='MultiClass', auto_class_weights='Balanced',
                                  task_type='GPU', verbose=0,
                                  l2_leaf_reg=3.0),
    }

X_pool = df_train[ALL_FEATURES].values
ret_pool = df_train['next_return'].values
prev_ret_pool = df_train['ret_1d'].fillna(0).values

tscv = TimeSeriesSplit(n_splits=5)
fold_results = []

for fold_idx, (tr_idx, te_idx) in enumerate(tscv.split(X_pool), start=1):
    # Compute labels using TRAIN-FOLD ONLY quantiles (no leakage)
    train_returns = pd.Series(ret_pool[tr_idx])
    _, lo, hi = make_labels(train_returns)
    all_returns = pd.Series(ret_pool)
    labels = pd.Series(1, index=all_returns.index)
    labels[all_returns > hi] = 2
    labels[all_returns < lo] = 0
    y_tr = labels.iloc[tr_idx].values
    y_te = labels.iloc[te_idx].values
    
    X_tr = X_pool[tr_idx]
    X_te = X_pool[te_idx]
    sw = compute_sample_weight('balanced', y_tr)
    
    models = make_models()
    models['xgb'].fit(X_tr, y_tr, sample_weight=sw)
    models['lgbm'].fit(X_tr, y_tr)
    models['cat'].fit(X_tr, y_tr)
    
    # Stacking — base probas on this fold's test as meta-input
    base_te = np.hstack([m.predict_proba(X_te) for m in models.values()])
    # Train meta on PREDICTIONS from a held-out portion of train to avoid leakage
    # Simplified: re-fit base on first 80% of train, predict on last 20%, that's meta-train
    split = int(len(X_tr) * 0.8)
    Xm_tr, Xm_val = X_tr[:split], X_tr[split:]
    ym_tr, ym_val = y_tr[:split], y_tr[split:]
    sw_m = compute_sample_weight('balanced', ym_tr)
    meta_models = make_models()
    meta_models['xgb'].fit(Xm_tr, ym_tr, sample_weight=sw_m)
    meta_models['lgbm'].fit(Xm_tr, ym_tr)
    meta_models['cat'].fit(Xm_tr, ym_tr)
    base_meta_train = np.hstack([m.predict_proba(Xm_val) for m in meta_models.values()])
    
    meta = LogisticRegression(max_iter=1000, class_weight='balanced', C=0.3)
    meta.fit(base_meta_train, ym_val)
    
    # Calibrate meta probabilities
    cal_meta = CalibratedClassifierCV(meta, method='sigmoid', cv='prefit')
    cal_meta.fit(base_meta_train, ym_val)
    
    y_pred = cal_meta.predict(base_te)
    y_proba = cal_meta.predict_proba(base_te)
    
    # Model accuracy + directional precision
    acc = accuracy_score(y_te, y_pred)
    dir_mask = y_pred != 1
    dir_prec = (y_pred[dir_mask] == y_te[dir_mask]).mean() if dir_mask.sum() else float('nan')
    
    # Baselines on same test slice
    prev_for_baseline = prev_ret_pool[te_idx]
    baselines = score_baselines(y_te, prev_for_baseline)
    
    fold_results.append({
        'fold': fold_idx,
        'train_n': len(tr_idx), 'test_n': len(te_idx),
        'model_acc': acc, 'model_dir_prec': dir_prec, 'model_n_dir': int(dir_mask.sum()),
        'baselines': baselines,
    })
    print(f'Fold {fold_idx}: train={len(tr_idx)} test={len(te_idx)} | '
          f'MODEL acc={acc:.3f} dir_prec={dir_prec:.3f} (n={dir_mask.sum()})')
    for bn, bv in baselines.items():
        print(f'    {bn:18s}: acc={bv["accuracy"]:.3f} dir_prec={bv["directional_precision"]:.3f} (n={bv["n_directional"]})')

In [ ]:
# Aggregate fold results
import pandas as pd
rows = []
for r in fold_results:
    rows.append({'fold': r['fold'], 'who': 'MODEL', 'acc': r['model_acc'], 'dir_prec': r['model_dir_prec']})
    for bn, bv in r['baselines'].items():
        rows.append({'fold': r['fold'], 'who': bn, 'acc': bv['accuracy'], 'dir_prec': bv['directional_precision']})
summary = pd.DataFrame(rows)
agg = summary.groupby('who').agg(
    mean_acc=('acc','mean'),
    std_acc=('acc','std'),
    mean_dir_prec=('dir_prec','mean'),
).sort_values('mean_dir_prec', ascending=False)
print('\n=== AVERAGED ACROSS 5 FOLDS ===')
print(agg.round(3))

## Final out-of-time test on the held-out 3 months
Train final models on all training pool, evaluate on never-seen holdout.

In [ ]:
X_full = df_train[ALL_FEATURES].values
ret_full = df_train['next_return']
_, lo, hi = make_labels(ret_full)
y_full = pd.Series(1, index=ret_full.index)
y_full[ret_full > hi] = 2
y_full[ret_full < lo] = 0
y_full = y_full.values
sw_full = compute_sample_weight('balanced', y_full)

final_models = make_models()
final_models['xgb'].fit(X_full, y_full, sample_weight=sw_full)
final_models['lgbm'].fit(X_full, y_full)
final_models['cat'].fit(X_full, y_full)

# Train meta on a held-out tail of training pool
split = int(len(X_full) * 0.8)
Xm_tr, Xm_val = X_full[:split], X_full[split:]
ym_tr, ym_val = y_full[:split], y_full[split:]
sw_m = compute_sample_weight('balanced', ym_tr)
meta_models = make_models()
meta_models['xgb'].fit(Xm_tr, ym_tr, sample_weight=sw_m)
meta_models['lgbm'].fit(Xm_tr, ym_tr)
meta_models['cat'].fit(Xm_tr, ym_tr)
base_meta_train = np.hstack([m.predict_proba(Xm_val) for m in meta_models.values()])
meta = LogisticRegression(max_iter=1000, class_weight='balanced', C=0.3)
meta.fit(base_meta_train, ym_val)
cal_meta = CalibratedClassifierCV(meta, method='sigmoid', cv='prefit')
cal_meta.fit(base_meta_train, ym_val)

# Apply labels to holdout using same thresholds from training pool
X_hold = df_holdout[ALL_FEATURES].values
ret_hold = df_holdout['next_return']
y_hold = pd.Series(1, index=ret_hold.index)
y_hold[ret_hold > hi] = 2
y_hold[ret_hold < lo] = 0
y_hold = y_hold.values

base_hold = np.hstack([m.predict_proba(X_hold) for m in final_models.values()])
y_pred_hold = cal_meta.predict(base_hold)
y_proba_hold = cal_meta.predict_proba(base_hold)

acc = accuracy_score(y_hold, y_pred_hold)
dir_mask = y_pred_hold != 1
dir_prec = (y_pred_hold[dir_mask] == y_hold[dir_mask]).mean() if dir_mask.sum() else 0
print(f'\n=== FINAL OUT-OF-TIME HOLDOUT ({len(df_holdout)} days, never seen during training) ===')
print(f'MODEL: acc={acc:.3f}  directional_prec={dir_prec:.3f}  ({dir_mask.sum()} directional preds)')

prev_hold = df_holdout['ret_1d'].fillna(0).values
hold_baselines = score_baselines(y_hold, prev_hold)
print('\nBaselines on same holdout:')
for bn, bv in hold_baselines.items():
    print(f'  {bn:18s}: acc={bv["accuracy"]:.3f}  dir_prec={bv["directional_precision"]:.3f}  (n={bv["n_directional"]})')

In [ ]:
# Confidence-precision diagnostic — is calibration working?
import numpy as np
max_proba = y_proba_hold.max(axis=1)
thresholds = [0.0, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
print('Confidence-gated precision on holdout (directional only):')
for t in thresholds:
    mask = (max_proba >= t) & (y_pred_hold != 1)
    if mask.sum() == 0:
        print(f'  conf>={t:.1f}: no trades')
        continue
    prec = (y_pred_hold[mask] == y_hold[mask]).mean()
    print(f'  conf>={t:.1f}: n={mask.sum():3d}  precision={prec:.3f}')

## Save models (only if they actually beat baselines)

In [ ]:
import joblib
# Save unconditionally; we'll judge from the printed metrics whether to use them.
joblib.dump(final_models['xgb'], f'{MODELS_DIR}/xgboost.pkl')
joblib.dump(final_models['lgbm'], f'{MODELS_DIR}/lightgbm.pkl')
joblib.dump(final_models['cat'], f'{MODELS_DIR}/catboost.pkl')
joblib.dump(cal_meta, f'{MODELS_DIR}/meta_rf.pkl')  # keep filename for backend compat

# Also save the label thresholds so we can label fresh data the same way
import json
with open(f'{MODELS_DIR}/label_thresholds.json', 'w') as f:
    json.dump({'q_low_return': float(lo), 'q_high_return': float(hi), 'features': ALL_FEATURES}, f)

print('\nSaved to', MODELS_DIR)
for f in sorted(os.listdir(MODELS_DIR)):
    size_mb = os.path.getsize(f'{MODELS_DIR}/{f}') / 1e6
    print(f'  {f}  ({size_mb:.2f} MB)')

## Decision guide

Look at the **directional precision** numbers (most important metric):

- **Model dir_prec > best baseline by >3 points** AND **dir_prec rises with confidence threshold** → ship it. Update backend.
- **Model dir_prec ≈ best baseline (within 2 points)** → model has marginal/no edge. Try Tier 2 (triple-barrier labels) or Tier 5 (India VIX).
- **Model dir_prec < best baseline** → trash this run, the features still don't separate signal from noise. Need different features.

If you ship: backend already has `FEATURE_COLUMNS` hardcoded in `feature_engineering.py`. You'll need to update that list to match `ALL_FEATURES` from this notebook, and update `add_indicators` etc. to produce them. I'll write that patch once you share the holdout numbers.